In [1]:
!git clone https://github.com/Santhosh-Analytics/transformer-from-scratch-en-fr.git
%cd transformer-from-scratch-en-fr

Cloning into 'transformer-from-scratch-en-fr'...
remote: Enumerating objects: 47, done.
remote: Counting objects: 100% (47/47), done.
remote: Compressing objects: 100% (36/36), done.
remote: Total 47 (delta 10), reused 44 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (47/47), 1.47 MiB | 19.48 MiB/s, done.
Resolving deltas: 100% (10/10), done.
/kaggle/working/transformer-from-scratch-en-fr


In [2]:
!pip install spacy sacrebleu pydantic-settings pyyaml --quiet
!python -m spacy download en_core_web_sm --quiet
!python -m spacy download fr_core_news_sm --quiet
!pip install -e . --no-deps --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 5.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 54.6 MB/s eta 0:00:0000:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.3/16.3 MB 69.6 MB/s eta 0:00:0000:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('fr_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
  Installing build dependencies ... done
  Checking if build back

In [3]:
!sed -i 's/pin_memory: false/pin_memory: true/' config/config.yaml
import torch
print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
print(f"Device   : {torch.cuda.get_device_name(0)}")

PyTorch  : 2.10.0+cu128
CUDA     : True
Device   : Tesla T4


In [4]:
!python -m src.components.data_ingestion
!python -m src.components.data_preprocessing
!python -m src.components.model

2026-03-27 20:28:45 | INFO     | transformer_mt | common.py:56 | Config loaded from: config/config.yaml
2026-03-27 20:28:45 | INFO     | transformer_mt | data_ingestion.py:82 | DataIngestion initialised.
2026-03-27 20:28:45 | INFO     | transformer_mt | data_ingestion.py:83 |   raw_dir  : artifacts/data/raw
2026-03-27 20:28:45 | INFO     | transformer_mt | data_ingestion.py:84 |   src_lang : en
2026-03-27 20:28:45 | INFO     | transformer_mt | data_ingestion.py:85 |   tgt_lang : fr
2026-03-27 20:28:45 | INFO     | transformer_mt | data_ingestion.py:130 | Raw data already present — skipping download.
2026-03-27 20:28:45 | INFO     | transformer_mt | data_ingestion.py:154 | ── Data Ingestion Summary ──────────────────────
2026-03-27 20:28:45 | INFO     | transformer_mt | data_ingestion.py:159 |   train.en  →  artifacts/data/raw/train.en  (29,000 lines)
2026-03-27 20:28:45 | INFO     | transformer_mt | data_ingestion.py:159 |   train.fr  →  artifacts/data/raw/train.fr  (29,000 lines)
2026

In [5]:
!python -m src.components.model_trainer

2026-03-27 20:29:48 | INFO     | transformer_mt | model_trainer.py:449 | Loading data …
2026-03-27 20:29:48 | INFO     | transformer_mt | common.py:56 | Config loaded from: config/config.yaml
2026-03-27 20:29:49 | INFO     | transformer_mt | data_preprocessing.py:167 | Loaded spaCy model: en_core_web_sm
2026-03-27 20:29:53 | INFO     | transformer_mt | data_preprocessing.py:167 | Loaded spaCy model: fr_core_news_sm
2026-03-27 20:29:53 | INFO     | transformer_mt | data_preprocessing.py:332 | DataPreprocessing initialised.
2026-03-27 20:29:53 | INFO     | transformer_mt | data_preprocessing.py:407 | ── Tokenising splits ───────────────────────────
2026-03-27 20:29:53 | INFO     | transformer_mt | data_preprocessing.py:340 | Loaded 29,000 sentences from artifacts/data/raw/train.en
2026-03-27 20:29:53 | INFO     | transformer_mt | data_preprocessing.py:340 | Loaded 29,000 sentences from artifacts/data/raw/train.fr
2026-03-27 20:29:53 | INFO     | transformer_mt | data_preprocessing.py:346

In [6]:
!sed -i 's/epochs: 20/epochs: 30/' config/config.yaml
!cat config/config.yaml
!python -m src.components.model_trainer

data:
  source: "multi30k"
  raw_dir: "artifacts/data/raw"
  processed_dir: "artifacts/data/processed"
  vocab_dir: "artifacts/vocab"
  src_lang: "en"
  tgt_lang: "fr"
  max_seq_len: 100
  min_freq: 2 # min word freq to include in vocab
model:
  d_model: 256
  num_heads: 8
  num_encoder_layers: 3
  num_decoder_layers: 3
  d_ff: 512 # feedforward dim (inner dim of FFN)
  dropout: 0.1
  max_seq_len: 100
training:
  batch_size: 128
  epochs: 30
  learning_rate: 0.0001
  clip_grad_norm: 1.0
  warmup_steps: 4000
  save_dir: "artifacts/models"
  model_name: "transformer_mt.pt"
  early_stopping_patience: 5
  pin_memory: true
evaluation:
  beam_size: 4
  bleu_max_n: 4
2026-03-27 20:44:17 | INFO     | transformer_mt | model_trainer.py:449 | Loading data …
2026-03-27 20:44:17 | INFO     | transformer_mt | common.py:56 | Config loaded from: config/config.yaml
2026-03-27 20:44:18 | INFO     | transformer_mt | data_preprocessing.py:167 | Loaded spaCy model: en_core_web_sm
2026-03-27 20:44:22 | INFO

In [7]:
!python -m src.components.model_evaluation

In [23]:
%%writefile main.py
"""
main.py
───────
Full pipeline entry point for the Transformer MT project.

Usage
─────
# Full pipeline: ingest → preprocess → train → evaluate
python main.py

# Skip ingestion if data already downloaded
python main.py --skip-ingest

# Resume training from checkpoint
python main.py --skip-ingest --resume

# Evaluate only (no training)
python main.py --skip-ingest --skip-train

# Translate custom sentences only
python main.py --skip-ingest --skip-train --skip-eval

Pipeline Stages
───────────────
1. DataIngestion      download Multi30k raw .txt files
2. DataPreprocessing  tokenise, build vocab, create DataLoaders
3. ModelTrainer       train Transformer with Noam LR + early stopping
4. ModelEvaluator     BLEU score on test set + sample translations
5. Translate          interactive custom sentence demo
"""

from __future__ import annotations

import argparse
import sys

import torch

from src.logger import logger
from src.utils.common import read_yaml
from src.components.data_ingestion import build_data_ingestion
from src.components.data_preprocessing import build_data_preprocessing
from src.components.model import build_transformer
from src.components.model_trainer import build_trainer
from src.components.model_evaluation import build_evaluator, translate


# ──────────────────────────────────────────────────────────────
# CLI
# ──────────────────────────────────────────────────────────────
def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Transformer EN→FR Machine Translation Pipeline"
    )
    parser.add_argument(
        "--skip-ingest",
        action="store_true",
        help="Skip data download (use if raw data already exists)",
    )
    parser.add_argument(
        "--skip-train",
        action="store_true",
        help="Skip training (use if checkpoint already exists)",
    )
    parser.add_argument(
        "--skip-eval",
        action="store_true",
        help="Skip evaluation",
    )
    parser.add_argument(
        "--resume",
        action="store_true",
        help="Resume training from existing checkpoint",
    )
    parser.add_argument(
        "--config",
        type=str,
        default="config/config.yaml",
        help="Path to config file (default: config/config.yaml)",
    )
    return parser.parse_args()


# ──────────────────────────────────────────────────────────────
# Pipeline stages
# ──────────────────────────────────────────────────────────────
def stage_ingest(config_path: str) -> None:
    logger.info("━━ Stage 1 / 4 : Data Ingestion ━━━━━━━━━━━━━━━")
    ingestion = build_data_ingestion(config_path)
    ingestion.run()


def stage_preprocess(config_path: str):
    logger.info("━━ Stage 2 / 4 : Data Preprocessing ━━━━━━━━━━━")
    pp = build_data_preprocessing(config_path)
    return pp.run()   # train_loader, val_loader, test_loader, src_vocab, tgt_vocab


def stage_train(
    train_loader,
    val_loader,
    src_vocab,
    tgt_vocab,
    config_path: str,
    resume: bool,
) -> None:
    logger.info("━━ Stage 3 / 4 : Model Training ━━━━━━━━━━━━━━━")
    cfg   = read_yaml(config_path)
    model = build_transformer(len(src_vocab), len(tgt_vocab), config_path)
    trainer = build_trainer(model, train_loader, val_loader, config_path)

    start_epoch = 1
    if resume:
        checkpoint = f"{cfg.training.save_dir}/{cfg.training.model_name}"
        start_epoch = trainer.load_checkpoint(checkpoint)

    history = trainer.run(start_epoch=start_epoch)

    # print loss curve summary
    print(f"\n{'Epoch':>6}  {'Train Loss':>10}  {'Val Loss':>10}  {'LR':>12}")
    print("─" * 46)
    for i, (tl, vl, lr) in enumerate(
        zip(history["train_loss"], history["val_loss"], history["lr"]),
        start=start_epoch,
    ):
        print(f"{i:>6}  {tl:>10.4f}  {vl:>10.4f}  {lr:>12.2e}")


def stage_evaluate(
    test_loader,
    src_vocab,
    tgt_vocab,
    config_path: str,
) -> dict:
    logger.info("━━ Stage 4 / 4 : Evaluation ━━━━━━━━━━━━━━━━━━━")
    cfg   = read_yaml(config_path)
    model = build_transformer(len(src_vocab), len(tgt_vocab), config_path)
    evaluator = build_evaluator(model, test_loader, src_vocab, tgt_vocab, config_path)
    checkpoint = f"{cfg.training.save_dir}/{cfg.training.model_name}"
    return evaluator.run(checkpoint)


def stage_translate(src_vocab, tgt_vocab, config_path: str) -> None:
    """Interactive translation demo with a few hardcoded examples."""
    logger.info("━━ Custom Translations ━━━━━━━━━━━━━━━━━━━━━━━━")
    cfg    = read_yaml(config_path)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # load best checkpoint into a fresh model
    from src.components.model_trainer import ModelTrainer
    model = build_transformer(len(src_vocab), len(tgt_vocab), config_path)
    checkpoint_path = f"{cfg.training.save_dir}/{cfg.training.model_name}"

    import torch as _torch
    ckpt = _torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(ckpt["model_state"])
    model = model.to(device)

    sentences = [
        "A dog is running in the park.",
        "Two children are playing with a ball.",
        "A woman is reading a book.",
        "The man is riding a bicycle.",
        "A group of people are sitting on the grass.",
        "A young girl is swimming in a pool.",
    ]

    print("\n── Custom sentence translations ──────────────────")
    for sent in sentences:
        fr = translate(sent, model, src_vocab, tgt_vocab, device)
        print(f"  EN : {sent}")
        print(f"  FR : {fr}")
        print()


# ──────────────────────────────────────────────────────────────
# Main
# ──────────────────────────────────────────────────────────────
def main() -> None:
    args = parse_args()

    logger.info("══════════════════════════════════════════════")
    logger.info("   Transformer EN→FR  —  Full Pipeline")
    logger.info("══════════════════════════════════════════════")
    logger.info(f"  config       : {args.config}")
    logger.info(f"  skip_ingest  : {args.skip_ingest}")
    logger.info(f"  skip_train   : {args.skip_train}")
    logger.info(f"  skip_eval    : {args.skip_eval}")
    logger.info(f"  resume       : {args.resume}")
    logger.info(f"  device       : {'cuda' if torch.cuda.is_available() else 'cpu'}")
    logger.info("══════════════════════════════════════════════")

    # ── stage 1: ingest ───────────────────────────────────────
    if not args.skip_ingest:
        stage_ingest(args.config)
    else:
        logger.info("Skipping data ingestion.")

    # ── stage 2: preprocess (always runs — builds DataLoaders) ─
    train_loader, val_loader, test_loader, src_vocab, tgt_vocab = \
        stage_preprocess(args.config)

    # ── stage 3: train ────────────────────────────────────────
    if not args.skip_train:
        stage_train(
            train_loader, val_loader,
            src_vocab, tgt_vocab,
            args.config, args.resume,
        )
    else:
        logger.info("Skipping training.")

    # ── stage 4: evaluate ─────────────────────────────────────
    if not args.skip_eval:
        scores = stage_evaluate(test_loader, src_vocab, tgt_vocab, args.config)
        print(f"\n{'─'*40}")
        print(f"  Final BLEU-4 : {scores['bleu']}")
        print(f"{'─'*40}\n")
    else:
        logger.info("Skipping evaluation.")

    # ── stage 5: translate demo ───────────────────────────────
    stage_translate(src_vocab, tgt_vocab, args.config)

    logger.info("══ Pipeline complete ══════════════════════════")


if __name__ == "__main__":
    main()

Overwriting main.py


In [19]:
!python -m src.components.evaluation

2026-03-27 21:16:04 | INFO     | transformer_mt | evaluation.py:406 | Loading data …
2026-03-27 21:16:04 | INFO     | transformer_mt | common.py:56 | Config loaded from: config/config.yaml
2026-03-27 21:16:05 | INFO     | transformer_mt | data_preprocessing.py:167 | Loaded spaCy model: en_core_web_sm
2026-03-27 21:16:09 | INFO     | transformer_mt | data_preprocessing.py:167 | Loaded spaCy model: fr_core_news_sm
2026-03-27 21:16:09 | INFO     | transformer_mt | data_preprocessing.py:332 | DataPreprocessing initialised.
2026-03-27 21:16:09 | INFO     | transformer_mt | data_preprocessing.py:407 | ── Tokenising splits ───────────────────────────
2026-03-27 21:16:09 | INFO     | transformer_mt | data_preprocessing.py:340 | Loaded 29,000 sentences from artifacts/data/raw/train.en
2026-03-27 21:16:09 | INFO     | transformer_mt | data_preprocessing.py:340 | Loaded 29,000 sentences from artifacts/data/raw/train.fr
2026-03-27 21:16:09 | INFO     | transformer_mt | data_preprocessing.py:346 | 